In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
uploaded = files.upload()  # contract_risk_augmented_v3.json 업로드

Saving contract_risk_augmented_v3.json to contract_risk_augmented_v3.json


In [ ]:
import json
from google.colab import userdata
from openai import OpenAI
from collections import Counter

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

SAVE_DIR = "/content/drive/Othercomputers/내 노트북/project/Workit"

with open(f"{SAVE_DIR}/data/jihye_sft/contract/augmented/contract_risk_augmented_v2.json", encoding="utf-8") as f:
    data = json.load(f)

cnt = Counter()
for d in data:
    verdict = d["output"].split("\n")[0].replace("판정:", "").strip()
    cnt[verdict] += 1

print(f"총 {len(data)}개")
for k, v in cnt.most_common():
    print(f"  {k}: {v}개 ({v/len(data)*100:.1f}%)")

normal = [d for d in data if "판정: 정상" in d["output"]]
print(f"\n기존 정상 샘플: {len(normal)}개")
print(f"\n샘플 예시:")
print(json.dumps(normal[0], ensure_ascii=False, indent=2))

총 196개
  위반: 70개 (35.7%)
  정상: 65개 (33.2%)
  누락: 61개 (31.1%)

기존 정상 샘플: 65개

샘플 예시:
{
  "instruction": "다음 계약 조항의 위험 여부를 판단하세요.",
  "input": "[계약조항]\n을이 계약기간 내에 용역을 완성하지 못한 경우 지체일수마다 계약금액의 0.1%를 지체상금으로 납부하여야 한다. 단, 지체상금의 총액은 계약금액의 30%를 초과할 수 없다.\n\n[참고기준]\n지방자치단체 용역계약 일반조건 제7절 제1항 가(행안부 예규 제324호): 계약서에 정한 지연배상금율에 계약금액을 곱하여 산출한 금액을 납부하여야 한다.\n지방계약법 시행령 제90조③: 지연배상금 총액은 계약금액의 100분의 30을 초과할 수 없다.",
  "output": "판정: 정상\n유형: 지체상금 상한 미설정\n근거: 지방계약법 시행령 제90조③에 따른 지체상금 30% 상한이 명시되어 있습니다. (관련 상위법령: 지방계약법 시행령 제90조③)"
}


In [ ]:
import json
import time
from collections import Counter
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

SAVE_DIR = "/content/drive/Othercomputers/내 노트북/project/Workit"

with open(f"{SAVE_DIR}/data/jihye_sft/contract/augmented/contract_risk_augmented_v3.json", encoding="utf-8") as f:
    data = json.load(f)

RISK_TYPES = [
    '추가 과업 비용 미지급', '손해배상 범위 일방적 제한', '대금 지급 기한 미설정',
    '갑의 일방적 해지권', '을의 해제권 배제/제한', '합의 없는 일방적 과업 추가',
    '지체상금률 과다 설정', '계약금액 조정 없는 과업변경', '지체상금 상한 미설정'
]

SYSTEM_PROMPT = """당신은 공공 SW 계약서의 위험 조항을 탐지하는 감리 전문가입니다.
주어진 계약 조항과 참고 기준을 바탕으로 위험 여부를 판단하고 근거를 제시하십시오."""

def generate_sample(verdict, risk_type):
    desc = {
        "위반": "법령 기준을 명백히 위반하는 조항 (기준값 초과, 권리 명시적 제한 등)",
        "누락": "법령 기준에서 요구하는 내용이 빠져있는 조항 (기한 미설정, 기준 미명시 등)"
    }
    user_prompt = f"""위험유형: {risk_type}
판정: {verdict} — {desc[verdict]}

새로운 계약 조항 예시를 만들어주세요.

출력 형식:
[계약조항]
(조항 내용)

[참고기준]
(관련 법령: 내용)

판정: {verdict}
유형: {risk_type}
근거: (판정 근거)"""

    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.9,
        max_tokens=500,
    )
    raw = resp.choices[0].message.content.strip()

    if "[계약조항]" not in raw or "[참고기준]" not in raw or f"판정: {verdict}" not in raw:
        return None

    contract = raw.split("[계약조항]")[1].split("[참고기준]")[0].strip()
    ref_and_output = raw.split("[참고기준]")[1].strip()
    ref = ref_and_output.split("판정:")[0].strip()
    output = "판정: " + ref_and_output.split("판정:")[1].strip()
    input_text = f"[계약조항]\n{contract}\n\n[참고기준]\n{ref}"

    return {
        "instruction": "다음 계약 조항의 위험 여부를 판단하세요.",
        "input": input_text,
        "output": output,
    }

existing_inputs = set(d["input"] for d in data)
new_samples = []

targets = {"위반": 80, "누락": 89}

for verdict, target in targets.items():
    count = 0
    i = 0
    while count < target:
        risk_type = RISK_TYPES[i % len(RISK_TYPES)]
        print(f"{verdict} [{count+1}/{target}] {risk_type}", end="\r")
        sample = generate_sample(verdict, risk_type)
        if sample and sample["input"] not in existing_inputs:
            new_samples.append(sample)
            existing_inputs.add(sample["input"])
            count += 1
        i += 1
        time.sleep(0.3)
    print(f"\n{verdict} 생성 완료: {count}개")

data_v4 = data + new_samples

out_path = f"{SAVE_DIR}/data/jihye_sft/contract/augmented/contract_risk_augmented_v4.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(data_v4, f, ensure_ascii=False, indent=2)

cnt = Counter()
for d in data_v4:
    verdict = d["output"].split("\n")[0].replace("판정:", "").strip()
    cnt[verdict] += 1

print(f"\n최종: {len(data_v4)}개")
for k, v in cnt.most_common():
    print(f"  {k}: {v}개 ({v/len(data_v4)*100:.1f}%)")


위반 생성 완료: 80개

누락 생성 완료: 89개

최종: 450개
  정상: 150개 (33.3%)
  누락: 150개 (33.3%)
  위반: 150개 (33.3%)


In [2]:
import json
import time
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

RAG_PATH = "/content/drive/Othercomputers/내 노트북/project/Workit/rag/pdfver_contract_review_output.json"
CONTRACT_PATH = "/content/drive/Othercomputers/내 노트북/project/Workit/data/jihye_sft/contract/augmented/contract_risk_augmented_v2.json"
OUT_PATH = "/content/drive/Othercomputers/내 노트북/project/Workit/data/jihye_sft/contract/augmented/contract_risk_augmented_v5.json"

with open(RAG_PATH, encoding="utf-8") as f:
    rag_data = json.load(f)

with open(CONTRACT_PATH, encoding="utf-8") as f:
    existing = json.load(f)

print(f"표준계약서 조항: {len(rag_data)}개")
print(f"기존 학습 데이터: {len(existing)}개")

SYSTEM_PROMPT = """당신은 공공 SW 계약서의 위험 조항을 탐지하는 감리 전문가입니다.
주어진 계약 조항과 참고 기준을 바탕으로 위험 여부를 판단하고 근거를 제시하십시오.

출력 형식:
판정: {정상/누락/위반}
유형: {위험유형명}
근거: {판정 근거}"""

def judge_clause(clause_text, law_refs):
    ref_text = "\n".join([
        f"{r['source_full']}: {r['chunk_text'][:200]}"
        for r in law_refs[:3]
    ])
    user_content = f"다음 계약 조항의 위험 여부를 판단하세요.\n\n[계약조항]\n{clause_text}\n\n[참고기준]\n{ref_text}"

    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        temperature=0.3,
        max_tokens=300,
    )
    return resp.choices[0].message.content.strip()

# 표준계약서 조항 판정
new_samples = []
for i, item in enumerate(rag_data):
    if not item["law_refs"]:
        continue

    print(f"[{i+1}/{len(rag_data)}] {item['clause_number']}", end="\r")
    output = judge_clause(item["clause_text"], item["law_refs"])

    # 정상인 경우만 학습 데이터로 추가
    if "판정: 정상" in output:
        ref_text = "\n".join([
            f"{r['source_full']}: {r['chunk_text'][:200]}"
            for r in item["law_refs"][:3]
        ])
        new_samples.append({
            "instruction": "다음 계약 조항의 위험 여부를 판단하세요.",
            "input": f"[계약조항]\n{item['clause_text']}\n\n[참고기준]\n{ref_text}",
            "output": output,
        })

    time.sleep(0.3)

print(f"\n정상 샘플 {len(new_samples)}개 추출")

# 기존 데이터에 추가
data_v5 = existing + new_samples

from collections import Counter
cnt = Counter()
for d in data_v5:
    verdict = d["output"].split("\n")[0].replace("판정:", "").strip()
    cnt[verdict] += 1

print(f"\n최종: {len(data_v5)}개")
for k, v in cnt.most_common():
    print(f"  {k}: {v}개 ({v/len(data_v5)*100:.1f}%)")

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data_v5, f, ensure_ascii=False, indent=2)
print(f"저장 완료 → {OUT_PATH}")

표준계약서 조항: 47개
기존 학습 데이터: 196개

정상 샘플 32개 추출

최종: 228개
  정상: 97개 (42.5%)
  위반: 70개 (30.7%)
  누락: 61개 (26.8%)
저장 완료 → /content/drive/Othercomputers/내 노트북/project/Workit/data/jihye_sft/contract/augmented/contract_risk_augmented_v5.json
